# Training the eyes of the Albion gathering bot

This notebook makes the `best.pt` file the bot needs to recognise trees.

You do not have to understand any of the code. Run the cells from top to bottom,
one at a time, with the round **play** button on the left of each one. Wait for a
green tick before starting the next.

**Do this first:** menu **Runtime -> Change runtime type -> T4 GPU -> Save**.
Without a GPU the training takes a whole day instead of about an hour.

## Step 1 - check the GPU is really on

If this prints a table with `Tesla T4` in it, you are set. If it says the command
is not found, the runtime type is still on CPU, go back and change it.

In [ ]:
!nvidia-smi

## Step 2 - install yolov5

This downloads the detection code and everything it needs. Takes two or three
minutes. Some red warnings about package versions are normal, ignore them.

In [ ]:
%cd /content
!git clone https://github.com/ultralytics/yolov5.git
%cd /content/yolov5

# The same version the bot on your PC uses, so the trained file loads there.
!git checkout --quiet f9e040b2c57e1db73f138ea420494fdeb05939da

!pip install --quiet -r requirements.txt roboflow

# wandb is an optional training dashboard that stops to ask for an account.
# The bot has no use for it, so it is removed before it can interrupt.
!pip uninstall --quiet -y wandb
%env WANDB_MODE=disabled
%env WANDB_SILENT=true

import torch
print()
print("torch", torch.__version__)
print("GPU  ", torch.cuda.get_device_name(0) if torch.cuda.is_available()
       else "NONE - go set Runtime > Change runtime type > T4 GPU")

## Step 3 - get the Albion screenshots

The pictures of trees and rocks live on Roboflow. They are free, but you need an
account to download them:

1. open <https://roboflow.com> and sign up, the free plan is enough
2. go to <https://app.roboflow.com/settings/api>
3. copy the **Private API Key**

Run the cell, paste the key in the box that appears, press Enter. The key stays
hidden as you type, that is normal.

In [ ]:
from getpass import getpass
from roboflow import Roboflow

API_KEY = getpass("Paste your Roboflow private API key here, then press Enter: ")

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("albiononline-c8fxi").project("albiongathering")

# Version 4 is the one the author of the bot trained on.
dataset = project.version(4).download("yolov5")

print()
print("Screenshots downloaded to:", dataset.location)

## Step 4 - point the config at the right folders and see the classes

Roboflow writes folder paths that do not resolve from here, so this fixes them.
It also prints what the model is about to learn.

Look at the list it prints. The bot turns those names into `tree`, `stone`, `ore`,
`fiber` and `hide` by matching words listed in `Application/Albion/resources.py`.
A tree class called something like `rought log` matches on `rought` and `log`, so
it works out of the box. If a name looks completely unrelated to any of those five,
tell Claude and it gets added to the alias list.

In [ ]:
from pathlib import Path
import yaml

location = Path(dataset.location)
data_yaml = location / "data.yaml"

config = yaml.safe_load(data_yaml.read_text())

config["train"] = str(location / "train" / "images")
config["val"] = str(location / "valid" / "images")

if (location / "test" / "images").exists():
    config["test"] = str(location / "test" / "images")

data_yaml.write_text(yaml.dump(config))

names = config["names"]
names = list(names.values()) if isinstance(names, dict) else names

print(f"{len(names)} classes this model will learn:")
for index, name in enumerate(names):
    print(f"  {index}: {name}")

for folder in ("train", "valid", "test"):
    images = location / folder / "images"
    if images.exists():
        print(f"{folder:6} {len(list(images.glob('*')))} images")

## Step 5 - train

This is the long one, roughly **40 to 70 minutes** on a T4. It prints a line per
epoch and slowly gets better.

Two things to know:

* **Do not close the browser tab.** Colab kills the runtime if you disconnect, and
  the training is lost. Leaving the tab open in the background is fine.
* The numbers to watch are at the right, `mAP50` in particular. Anything above
  `0.8` makes a bot that finds trees reliably.

`--patience 25` stops early if it stops improving, so it may well finish sooner.

In [ ]:
!python train.py \
  --img 640 \
  --batch 16 \
  --epochs 100 \
  --patience 25 \
  --data {dataset.location}/data.yaml \
  --weights yolov5s.pt \
  --name albion \
  --cache

## Step 6 - look at whether it actually learned

The curves should climb and flatten out. The bottom right two, `mAP50` and
`mAP50-95`, are the ones that matter, they should end up high and stop wobbling.

In [ ]:
from IPython.display import Image, display

run = "/content/yolov5/runs/train/albion"

display(Image(f"{run}/results.png", width=1000))
display(Image(f"{run}/confusion_matrix.png", width=700))

## Step 7 - see it detect trees on real screenshots

Boxes drawn around the trees means it works. If the pictures come back empty, the
training did not take and it is worth running step 5 again with more epochs.

In [ ]:
from IPython.display import Image, display
from pathlib import Path

!python detect.py --weights {run}/weights/best.pt --img 640 --conf 0.5 \
    --source {dataset.location}/valid/images --name check --exist-ok

for picture in sorted(Path("/content/yolov5/runs/detect/check").glob("*.jpg"))[:4]:
    display(Image(str(picture), width=600))

## Step 8 - download best.pt

This drops `best.pt` in your Windows **Downloads** folder. Your browser may ask
you to allow the download, say yes.

Once it is there, tell Claude and it puts the file where the bot expects it.

In [ ]:
import shutil
from google.colab import files

shutil.copy(f"{run}/weights/best.pt", "/content/best.pt")

size = round(shutil.os.path.getsize("/content/best.pt") / 1e6, 1)
print(f"best.pt is {size} MB, downloading it now...")

files.download("/content/best.pt")